In [97]:
import os
import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
API_TOKEN = os.getenv("TOGGL_API_TOKEN")

# 抽出期間の設定（過去7日間）
end_date = datetime.now()
start_date = end_date - timedelta(days=7)
START_DATE_STR = start_date.strftime('%Y-%m-%d')
END_DATE_STR = end_date.strftime('%Y-%m-%d')


# 1. 新ルートでプロジェクトマスタを辞書型として取得
PROJECTS_URL = 'https://api.track.toggl.com/api/v9/me/projects'
res_pj = requests.get(PROJECTS_URL, auth=HTTPBasicAuth(API_TOKEN, 'api_token'))

project_mapping = {}
if res_pj.status_code == 200:
    for pj in res_pj.json():
        # { 123456789: "本業", 987654321: "副業" } というマッピング辞書を自動生成
        project_mapping[pj.get('id')] = pj.get('name')

# 2. タイムログ（Time Entries）を取得
ENTRIES_URL = 'https://api.track.toggl.com/api/v9/me/time_entries'
# ※必要に応じて日付パラメータ（start_date, end_date）を追加してください
res_entries = requests.get(ENTRIES_URL, auth=HTTPBasicAuth(API_TOKEN, 'api_token'))

if res_entries.status_code == 200:
    entries = res_entries.json()
    
    # Pandasデータフレームに変換
    df = pd.DataFrame(entries)
    # データの加工
    df['start'] = pd.to_datetime(df['start'])
    df['date'] = df['start'].dt.strftime('%Y-%m-%d')
    df['duration_minutes'] = (df['duration'] / 60).round(1)
    df = df[df['duration'] > 0]
    df['tags'] = df['tags'].astype(str).str.replace("#","").str.replace("[","").str.replace("]","").str.replace("'","")
    df['start'] = df["start"].dt.tz_convert('Asia/Tokyo')
    
    # 3. 冷徹にプロジェクト名を紐付ける（ここが今回の心臓部です）
    # タイムログ側の 'project_id' をキーにして、先ほどの辞書からプロジェクト名を引っ張る
    # 辞書になければ、安全に「プロジェクトなし」を返す
    df['project_name'] = df['project_id'].map(project_mapping).fillna('プロジェクトなし')
    df['project_name'] =df['project_name'].str.split("_", expand = True)[1]
    # 4. 必要な列だけを綺麗に並び替えて抽出（Looker Studio用）
    # ※必要な項目（開始時間、所要時間、タスク名など）を絞り込んでください
    result_df = df[['start', 'tags', 'project_name','duration_minutes']]
    result_df["年"] = result_df["start"].dt.year
    result_df["月"] = result_df["start"].dt.month
    result_df["時"] = result_df["start"].dt.hour
    result_df['時間帯'] = ['朝' if 6 <= h <= 10 else '昼' if 11 <= h <= 14 else '夕方' if 15 <= h <= 18 else '夜' if 19 <= h <= 22 else '深夜' for h in result_df['時']]
    print("✅ データの整形に成功しました。")
    print(result_df.head(10)) # 先頭10件を画面に確認出力
    
    # この後、result_df をスプレッドシートへの書き込み処理へ渡します。
else:
    print(f"タイムログの取得に失敗しました: {res_entries.text}")

タイムログの取得に失敗しました: Incorrect username and/or password


In [93]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# =========================================================================
# 1. 認証・設定（Toggl & Google API）
# =========================================================================
API_TOKEN = 'TOGGL_API_TOKEN'
auth_config = HTTPBasicAuth(API_TOKEN, 'api_token')

# 【Google APIの設定】
# ダウンロードしたJSON鍵ファイル名と、書き込み先スプレッドシート名を設定してください
GOOGLE_KEY_FILE = 'credentials.json' 
SPREADSHEET_NAME = 'Toggl' # スプレッドシート名
WORKSHEET_NAME = 'シート1'                 # シート名（必要に応じて変更）


# =========================================================================
# 2. プロジェクトマスタ（IDと名前の対応表）の取得
# =========================================================================
PROJECTS_URL = 'https://api.track.toggl.com/api/v9/me/projects'
project_mapping = {}

try:
    res_pj = requests.get(PROJECTS_URL, auth=auth_config)
    if res_pj.status_code == 200:
        for pj in res_pj.json():
            project_mapping[pj.get('id')] = pj.get('name')
except Exception as e:
    print(f"❌ プロジェクト取得通信エラー: {e}")


# =========================================================================
# 3. タイムログ（Time Entries）の取得
# =========================================================================
ENTRIES_URL = 'https://api.track.toggl.com/api/v9/me/time_entries'
query_params = {} # 必要に応じて期間指定パラメータを追加

try:
    res_entries = requests.get(ENTRIES_URL, auth=auth_config, params=query_params)
    
    if res_entries.status_code == 200:
        entries_json = res_entries.json()
        df = pd.DataFrame(entries_json)
        
        if not df.empty:
            # マッピングと整形
            df['project_name'] = df['project_id'].map(project_mapping).fillna('プロジェクトなし')
            result_df = df[['start', 'description', 'project_name', 'duration']]
            result_df.columns = ['開始時間', 'タスク名', 'プロジェクト名', '所要時間(秒)']
            
            print(f"✅ 【加工完了】最新データ {len(result_df)} 件を検知。")
            
            # =========================================================================
            # 4. Googleスプレッドシートへの自動上書き（中継フェーズ）
            # =========================================================================
            print("\n【蓄積】Googleスプレッドシートへの書き込みを開始します...")
            
            # API認証の確立
            scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
            creds = ServiceAccountCredentials.from_json_keyfile_name(GOOGLE_KEY_FILE, scope)
            gc = gspread.authorize(creds)
            
            # シートの展開
            sh = gc.open(SPREADSHEET_NAME)
            worksheet = sh.worksheet(WORKSHEET_NAME)
            
            # 💡 データの整合性を保つため、一度シートの中身を完全にクリーンにする
            worksheet.clear()
            
            # Pandasのデータフレームを、gspreadが扱える「ヘッダー入りの2次元リスト」に一発変換
            # NaN（欠損値）を空文字に置換して、スプレッドシート側のパースエラーを徹底防衛
            upload_data = [result_df.columns.values.tolist()] + result_df.fillna('').values.tolist()
            
            # セル「A1」を起点に、すべてのデータを一括更新
            worksheet.update('A1', upload_data)
            
            print(f"✅ 【蓄積完了】スプレッドシート「{SPREADSHEET_NAME}」へ正常に上書きされました。")
            print("これでLooker Studio側の参照元データが最新状態に同期されました。")
            
        else:
            print("⚠️ タイムログが空のため、スプレッドシートの更新をスキップしました。")
    else:
        print(f"❌ タイムログ取得失敗：{res_entries.text}")

except Exception as e:
    print(f"❌ 処理中にエラーが発生しました（Google認証、または通信）: {e}")

❌ タイムログ取得失敗：Incorrect username and/or password
